In [ ]:
# RabindricGPT — Stage 1 Ingestion, Shard Build & Drive Preparation on CPU Colab
# Streams FineWeb-2 Bengali dataset up to 1.1 GB in resumable chunks, filters Tagore leakage, builds shards and prepares Drive.

import os
import sys
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive mount skipped: {e}")

# 2. Sync Repository
repo_path = Path('/content/RabindricGPT')
if not repo_path.exists():
    !git clone https://github.com/ASMITSARKAR/RabindricGPT.git /content/RabindricGPT
else:
    !git -C /content/RabindricGPT pull --ff-only

os.chdir('/content/RabindricGPT')

# 3. Install dependencies
!pip install -q pyarrow fsspec sentencepiece pyyaml

# 4. Run Stage 1 Ingest on CPU runtime to 1.1 GB target
!python scripts/02_ingest_fineweb2.py --target-bytes 1100000000 --egen-chars 100000

# 5. Build shards across all 5 funded tokenizers
!python scripts/07_build_shards.py

# 6. Package prepared/ directory and update SHA-256 hashes for Drive
!python scripts/audits/build_prepared_drive.py

# 7. Sync prepared/ to Google Drive
drive_prep = Path('/content/drive/MyDrive/RabindricGPT/prepared')
drive_prep.mkdir(parents=True, exist_ok=True)
!cp -r prepared/* /content/drive/MyDrive/RabindricGPT/prepared/
print('Stage 1 data and prepared shards synced to Drive successfully!')